# Train the Checker (Stage 2)

Fine-tunes a `yolo26-cls` binary classifier (`ashwagandha` vs.
`not_ashwagandha`) on the crops Stage 1 hands it. Same checkpointing/resume
setup as `train_finder.ipynb` -- Ultralytics saves `last.pt` every epoch
regardless of task, so this is safe to interrupt and re-run.

Needs `scripts/prepare_classifier_data.py` to have already been run (needs
`data/raw/negatives/<source>/` to have images in it first).

**Two numbers below, same as the Finder notebook:** the val-accuracy cell is
what training itself watches (it's literally what `best.pt` gets picked
against, so treat it as informative but optimistic, not neutral), and a
separate test-set cell further down gives the honest, never-touched-during-
training number -- same reasoning `train_finder.ipynb` already uses for
Stage 1's test-set check.

In [ ]:
from pathlib import Path

from ultralytics import YOLO

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = REPO_ROOT / "data" / "classify"
MODEL_SIZE = "m"    # only matters for a fresh start -- there's no domain-specific prior
                     # checkpoint for the classifier the way best_m.pt exists for the Finder,
                     # so a fresh run always starts from Ultralytics' generic pretrained weights
STARTING_WEIGHTS = f"yolo26{MODEL_SIZE}-cls.pt"
PROJECT_DIR = REPO_ROOT / "runs" / "checker"
RUN_NAME = "train"

EPOCHS = 60
PATIENCE = 20       # unlike Stage 1's patience value, this one isn't backed by an actual
                     # observed training curve yet -- adjust if you see it stop too early or too late
IMGSZ = 224
BATCH = -1           # -1 = let Ultralytics pick the biggest batch that fits your GPU
AUTO_AUGMENT = "randaugment"   # small dataset -- fight overfitting with strong augmentation,
                                 # same reasoning as the Azadnia et al. 2024 paper the README cites
SAVE_PERIOD = 10    # keep a numbered snapshot (epoch10.pt, epoch20.pt, ...) every N epochs too

In [ ]:
last_checkpoint = PROJECT_DIR / RUN_NAME / "weights" / "last.pt"


def start_fresh():
    for split in ("train", "val", "test"):
        split_dir = DATA_DIR / split
        if not split_dir.is_dir() or not any(split_dir.iterdir()):
            raise SystemExit(
                f"{split_dir} is missing or empty. Run scripts/prepare_classifier_data.py first."
            )
    model = YOLO(STARTING_WEIGHTS)
    model.train(
        data=str(DATA_DIR),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        auto_augment=AUTO_AUGMENT,
        save_period=SAVE_PERIOD,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
    )
    return model


if last_checkpoint.exists():
    try:
        print(f"found a checkpoint at {last_checkpoint} -- trying to resume it")
        model = YOLO(str(last_checkpoint))
        # pass the checkpoint PATH here, not just True -- see train_finder.ipynb for why
        model.train(resume=str(last_checkpoint))
    except AssertionError as e:
        print(f"nothing to resume ({e})")
        print("starting a fresh run instead -- it'll land in a new numbered folder")
        model = start_fresh()
else:
    print("no existing checkpoint found, starting fresh")
    model = start_fresh()

save_dir = model.trainer.save_dir
print(f"\nthis run's files are in: {save_dir}")

### check the result

`model.val()` re-runs validation cleanly and prints top-1 accuracy directly,
rather than trusting whatever scrolled by during training.

In [ ]:
# project/name here matter -- without them Ultralytics saves to a default location
# based on wherever this notebook's working directory happens to be (notebooks/, not
# the repo root), scattering results away from everything else about this run
val_metrics = model.val(data=str(DATA_DIR), imgsz=IMGSZ, project=str(save_dir), name="val_check")
print(f"\nval top-1 accuracy: {val_metrics.top1:.3f}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


def show(path, title):
    if not path.exists():
        print(f"(missing: {path})")
        return
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(path))
    plt.title(title)
    plt.axis("off")
    plt.show()


show(save_dir / "results.png", "training curves (all epochs)")
show(save_dir / "confusion_matrix.png", "confusion matrix (val split)")
show(save_dir / "val_batch0_labels.jpg", "val split -- ground truth")
show(save_dir / "val_batch0_pred.jpg", "val split -- what the model predicted")

### test set



In [ ]:
test_metrics = model.val(data=str(DATA_DIR), split="test", imgsz=IMGSZ,
                          project=str(save_dir), name="test_check")
print(f"\ntest top-1 accuracy: {test_metrics.top1:.3f}")

test_dir = save_dir / "test_check"
show(test_dir / "confusion_matrix.png", "confusion matrix (test split)")